In [1]:
import os
from google.colab import files

# Create a clean directory for evidence
!mkdir -p /content/evidence
%cd /content/evidence

# Upload the log files (auth.log, wtmp, etc.)
print("Upload your log files (auth.log, wtmp, etc.) below:")
uploaded = files.upload()

# List files to confirm
!ls -lh

/content/evidence
Upload your log files (auth.log, wtmp, etc.) below:


Saving Microsoft-Windows-Sysmon-Operational.evtx to Microsoft-Windows-Sysmon-Operational.evtx
total 1.1M
-rw-r--r-- 1 root root 1.1M Dec 19 22:22 Microsoft-Windows-Sysmon-Operational.evtx


In [2]:
# Setup: Convert EVTX to CSV using EvtxECmd
# Replicates the devenv.nix setup for Colab environment

import os
import subprocess
from pathlib import Path

# Download and setup EvtxECmd if not already done
evtx_dir = Path('.evtxecmd')
if not evtx_dir.exists():
    print("Setting up EvtxECmd...")
    evtx_dir.mkdir(exist_ok=True)

    # Download EvtxECmd
    !curl -L https://download.ericzimmermanstools.com/net6/EvtxECmd.zip -o .evtxecmd/EvtxECmd.zip
    !unzip -q .evtxecmd/EvtxECmd.zip -d .evtxecmd
    !rm .evtxecmd/EvtxECmd.zip
    print("EvtxECmd downloaded and extracted")

# Install dotnet runtime if needed (Colab doesn't have it by default)
try:
    subprocess.run(['dotnet', '--version'], capture_output=True, check=True)
    print("Dotnet runtime already installed")
except:
    print("Installing dotnet runtime...")
    !wget https://dot.net/v1/dotnet-install.sh
    !chmod +x dotnet-install.sh
    !./dotnet-install.sh --channel 6.0 --runtime dotnet --install-dir /usr/share/dotnet
    os.environ['PATH'] = f"/usr/share/dotnet:{os.environ['PATH']}"
    print("Dotnet runtime installed")

# Convert EVTX to CSV
# Adjust the input filename as needed
evtx_file = "Microsoft-Windows-Sysmon-Operational.evtx"
output_dir = "csv_output"

print(f"\nConverting {evtx_file} to CSV...")
!dotnet .evtxecmd/EvtxeCmd/EvtxECmd.dll -f {evtx_file} --csv {output_dir} --csvf sysmon_events.csv

print(f"\nConversion complete! CSV output in {output_dir}/")
print(f"Loading sysmon_events.csv for analysis...")

Setting up EvtxECmd...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 4245k  100 4245k    0     0  7458k      0 --:--:-- --:--:-- --:--:-- 7448k
EvtxECmd downloaded and extracted
Installing dotnet runtime...
--2025-12-19 22:25:40--  https://dot.net/v1/dotnet-install.sh
Resolving dot.net (dot.net)... 20.76.201.171, 20.112.250.133, 20.236.44.162, ...
Connecting to dot.net (dot.net)|20.76.201.171|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh [following]
--2025-12-19 22:25:41--  https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh
Resolving builds.dotnet.microsoft.com (builds.dotnet.microsoft.com)... 23.192.229.105, 23.192.229.109, 2600:1408:5400:13::17cf:caaf, ...
Connecting to builds.dotnet.microsoft.com (builds.dotnet.microsoft.com)|23.192.

In [5]:
# Task 1: How many Event ID 11 logs exist?
# VD workflow: Navigate to EventId column → Shift+F for frequency histogram
# Question: Event ID 11 indicates file creation - how many occurred?

import pandas as pd
from itables import show

# Load the converted Sysmon event data
df = pd.read_csv('csv_output/sysmon_events.csv')

print(f"Total events loaded: {len(df)}")

# Frequency analysis on EventId - replicates VD Shift+F
# This histogram view immediately reveals the count for each event type
eventid_freq = df['EventId'].value_counts().reset_index()
eventid_freq.columns = ['EventId', 'Count']
eventid_freq = eventid_freq.sort_values('EventId').reset_index(drop=True)  # Clean up index

show(eventid_freq, caption="Event ID Frequency Distribution - EventId 11 count visible immediately")

# Highlight the answer
answer = eventid_freq[eventid_freq['EventId'] == 11]['Count'].values[0]
print(f"\n✓ Answer: {answer} events with EventId 11 (File Creation events)")

Total events loaded: 169


Loading ITables v2.6.1 from the internet... (need help?)



✓ Answer: 56 events with EventId 11 (File Creation events)


In [8]:
# Task 2: What is the malicious process that infected the system?
# VD workflow: From Task 1 frequency view → scroll to EventId 1 → press enter to filter
#              → navigate to ExecutableInfo column → spot "cyberjunkie" in path
# Question: Event ID 1 = Process Creation - which executable looks suspicious?

# Filter to EventId 1 (Process Creation events) - only 6 events, easy to scan
eventid_1 = df[df['EventId'] == 1].copy()

print(f"EventId 1 (Process Creation) events: {len(eventid_1)}")

# Show key fields for process analysis
# ExecutableInfo should reveal the suspicious process path
process_view = eventid_1[['TimeCreated', 'ExecutableInfo', 'PayloadData1', 'PayloadData2']].copy()

show(process_view, caption="Process Creation Events - ExecutableInfo shows suspicious path")

# Isolate the suspicious entry - looking for Downloads folder, unusual naming
# Based on writeup: double .exe extension, from CyberJunkie\Downloads
suspicious = eventid_1[eventid_1['ExecutableInfo'].str.contains('Downloads', na=False)]

if len(suspicious) > 0:
    show(suspicious[['TimeCreated', 'ExecutableInfo']],
         caption="Suspicious Process - Executed from Downloads folder")

    answer = suspicious['ExecutableInfo'].values[0]
    print(f"\n✓ Answer: {answer}")
    print("✓ Red flags: Double .exe.exe extension, executed from Downloads, unusual name")

EventId 1 (Process Creation) events: 6


Loading ITables v2.6.1 from the internet... (need help?)


Loading ITables v2.6.1 from the internet... (need help?)



✓ Answer: "C:\Users\CyberJunkie\Downloads\Preventivo24.02.14.exe.exe" 
✓ Red flags: Double .exe.exe extension, executed from Downloads, unusual name


In [9]:
# Task 3: Which cloud drive was used to distribute the malware?
# VD workflow: Navigate to TimeCreated → press [ for oldest→newest sort
#              → press on first entry → see the name in PayloadData6
# Question: DNS queries reveal domains contacted - which cloud service?

# Sort entire dataset by TimeCreated (oldest first) - replicates VD [ command
df_sorted = df.sort_values('TimeCreated').reset_index(drop=True)

print("Showing earliest events chronologically:")

# Display first several events to see the initial activity
earliest_events = df_sorted.head(10)[['TimeCreated', 'EventId', 'PayloadData6', 'ExecutableInfo']].copy()

show(earliest_events, caption="Earliest Events (Oldest→Newest) - Cloud service visible in PayloadData6")

# Focus on the very first entry as per VD workflow
first_event = df_sorted.iloc[0]
print(f"\n✓ First event details:")
print(f"   TimeCreated: {first_event['TimeCreated']}")
print(f"   EventId: {first_event['EventId']}")
print(f"   PayloadData6: {first_event['PayloadData6']}")

# Extract answer from PayloadData6
if pd.notna(first_event['PayloadData6']) and 'dropbox' in str(first_event['PayloadData6']).lower():
    print(f"\n✓ Answer: dropbox")
    print("✓ Cloud service used for malware distribution visible in earliest DNS query")

Showing earliest events chronologically:


Loading ITables v2.6.1 from the internet... (need help?)



✓ First event details:
   TimeCreated: 2024-02-14 03:41:26.4441194
   EventId: 22
   PayloadData6: QueryResults: type:  5 edge-block-www-env.dropbox-dns.com;::ffff:162.125.81.15;198.51.44.6;2620:4d:4000:6259:7:6:0:1;198.51.45.6;2a00:edc0:6259:7:6::2;198.51.44.70;2620:4d:4000:6259:7:6:0:3;198.51.45.70;2a00:edc0:6259:7:6::4;

✓ Answer: dropbox
✓ Cloud service used for malware distribution visible in earliest DNS query


In [10]:
# Task 4: What timestamp was the PDF file changed to (timestomping)?
# VD workflow: EventId frequency → select EventId 2 → navigate to PayloadData4
#              → filter for "pdf" (|) → isolate (") → check PayloadData5
# Question: EventId 2 = File creation time changed - what was PDF stomped to?

# Start with frequency view context (we saw this in Task 1)
print("EventId 2 = File creation time changed (Timestomping)")

# Filter to EventId 2 (Timestomping events)
eventid_2 = df[df['EventId'] == 2].copy()

print(f"EventId 2 events found: {len(eventid_2)}\n")

# Show all timestomping events to understand scope
show(eventid_2[['TimeCreated', 'PayloadData1', 'PayloadData4', 'PayloadData5', 'ExecutableInfo']],
     caption="File Creation Time Changed Events - Searching for PDF")

# Filter PayloadData4 for PDF files - replicates VD | search then " isolate
pdf_timestomp = eventid_2[eventid_2['PayloadData4'].str.contains('pdf', case=False, na=False)].copy()

print(f"\nPDF file timestomping events: {len(pdf_timestomp)}")

# Show the isolated PDF entry - PayloadData5 contains the answer (new timestamp)
show(pdf_timestomp[['PayloadData4', 'PayloadData5', 'ExecutableInfo']],
     caption="PDF File Timestomped - PayloadData5 shows the fake timestamp")

if len(pdf_timestomp) > 0:
    answer = pdf_timestomp.iloc[0]['PayloadData5']
    print(f"\n✓ Answer: {answer}")
    print("✓ This is the fabricated timestamp the attacker set to evade detection")

EventId 2 = File creation time changed (Timestomping)
EventId 2 events found: 16



Loading ITables v2.6.1 from the internet... (need help?)



PDF file timestomping events: 1


Loading ITables v2.6.1 from the internet... (need help?)



✓ Answer: CreationTimeUTC: 2024-01-14 08:10:06.029
✓ This is the fabricated timestamp the attacker set to evade detection


In [11]:
# Task 5: Where was once.cmd created on disk?
# VD workflow: Navigate to PayloadData4 → | search for "once.cmd" → " isolate
#              → find entry with EventId 11 → navigate to PayloadData4 for full path
# Question: EventId 11 = File created - what's the full path to once.cmd?

# Search PayloadData4 for once.cmd across all events
once_cmd_events = df[df['PayloadData4'].str.contains('once.cmd', case=False, na=False)].copy()

print(f"Events containing 'once.cmd': {len(once_cmd_events)}")

# Show all once.cmd references to see context
show(once_cmd_events[['EventId', 'TimeCreated', 'PayloadData4', 'ExecutableInfo']],
     caption="Events referencing once.cmd - Multiple EventIds may appear")

# Isolate EventId 11 (File Created) as per VD workflow
once_cmd_created = once_cmd_events[once_cmd_events['EventId'] == 11].copy()

print(f"\nEventId 11 (File Created) entries for once.cmd: {len(once_cmd_created)}")

# Show the File Creation event with full path in PayloadData4
show(once_cmd_created[['TimeCreated', 'PayloadData4', 'ExecutableInfo']],
     caption="once.cmd File Creation Event - Full path in PayloadData4")

if len(once_cmd_created) > 0:
    answer = once_cmd_created.iloc[0]['PayloadData4']
    print(f"\n✓ Answer: {answer}")
    print("✓ Full path where malicious batch file was dropped")

Events containing 'once.cmd': 4


Loading ITables v2.6.1 from the internet... (need help?)



EventId 11 (File Created) entries for once.cmd: 2


Loading ITables v2.6.1 from the internet... (need help?)



✓ Answer: TargetFilename: C:\Users\CyberJunkie\AppData\Roaming\Photo and Fax Vn\Photo and vn 1.1.2\install\F97891C\WindowsVolume\Games\once.cmd
✓ Full path where malicious batch file was dropped


In [13]:
# Task 6: What dummy domain did the malicious file try to connect to?
# VD workflow: Back to Task 1 frequency view → select EventId 22 (DNS Query)
#              → navigate to PayloadData3 (Image field) → | search "Prevent" → " isolate
#              → check PayloadData for domain name
# Question: EventId 22 = DNS Query - which domain was queried by malicious process?

# Start with EventId 22 (DNS Query events)
eventid_22 = df[df['EventId'] == 22].copy()

print(f"EventId 22 (DNS Query) events: {len(eventid_22)}")

# Show all DNS queries with more PayloadData fields to find domain
show(eventid_22[['TimeCreated', 'PayloadData3', 'PayloadData4', 'PayloadData5', 'PayloadData6']],
     caption="DNS Query Events - Scanning PayloadData fields for domain name")

# Filter PayloadData3 for "Prevent" - the malicious executable
prevent_dns = eventid_22[eventid_22['PayloadData3'].str.contains('Prevent', case=False, na=False)].copy()

print(f"\nDNS queries from Preventivo executable: {len(prevent_dns)}")

# Show all PayloadData fields to identify which contains the domain
show(prevent_dns,
     caption="Preventivo DNS Query - Full event details")

if len(prevent_dns) > 0:
    # Check PayloadData4 or PayloadData6 for domain (QueryName field)
    row = prevent_dns.iloc[0]
    print(f"\nPayloadData fields:")
    for col in ['PayloadData1', 'PayloadData2', 'PayloadData3', 'PayloadData4', 'PayloadData5', 'PayloadData6']:
        if col in row and pd.notna(row[col]):
            print(f"  {col}: {row[col]}")

EventId 22 (DNS Query) events: 3


Loading ITables v2.6.1 from the internet... (need help?)



DNS queries from Preventivo executable: 1


Loading ITables v2.6.1 from the internet... (need help?)



PayloadData fields:
  PayloadData1: ProcessID: 10672, ProcessGUID: 817bddf3-3684-65cc-2d02-000000001900
  PayloadData2: RuleName: -
  PayloadData3: Image: C:\Users\CyberJunkie\Downloads\Preventivo24.02.14.exe.exe
  PayloadData4: QueryName: www.example.com
  PayloadData5: QueryStatus: 0
  PayloadData6: QueryResults: ::ffff:93.184.216.34;199.43.135.53;2001:500:8f::53;199.43.133.53;2001:500:8d::53;


In [14]:
# Task 7: Which IP address did the malicious process try to reach?
# VD workflow: Navigate to EventId column → Shift+F → select EventId 3
#              → only one event, answer immediately visible
# Question: EventId 3 = Network Connection - what destination IP?

# Filter to EventId 3 (Network Connection events)
eventid_3 = df[df['EventId'] == 3].copy()

print(f"EventId 3 (Network Connection) events: {len(eventid_3)}")
print("Only one event - answer immediately visible\n")

# Show the single network connection event with all relevant fields
show(eventid_3[['TimeCreated', 'ExecutableInfo', 'PayloadData1', 'PayloadData2',
                'PayloadData3', 'PayloadData4', 'PayloadData5', 'PayloadData6']],
     caption="Network Connection Event - Destination IP in payload data")

# Show full event for complete context
show(eventid_3, caption="Complete Network Connection Details")

if len(eventid_3) > 0:
    row = eventid_3.iloc[0]
    # Destination IP is likely in one of the PayloadData fields
    print(f"\n✓ Single network connection from malicious process")
    print(f"✓ Scan PayloadData fields for destination IP address")

EventId 3 (Network Connection) events: 1
Only one event - answer immediately visible



Loading ITables v2.6.1 from the internet... (need help?)


Loading ITables v2.6.1 from the internet... (need help?)



✓ Single network connection from malicious process
✓ Scan PayloadData fields for destination IP address


In [15]:
# Task 8: When did the malicious process terminate itself?
# VD workflow: Navigate to EventId column → Shift+F → select EventId 5
#              → only one event here as well :)
# Question: EventId 5 = Process Termination - when did Preventivo terminate?

# Filter to EventId 5 (Process Termination events)
eventid_5 = df[df['EventId'] == 5].copy()

print(f"EventId 5 (Process Termination) events: {len(eventid_5)}")
print("Only one event - answer immediately visible\n")

# Show the single process termination event
show(eventid_5[['TimeCreated', 'ExecutableInfo', 'PayloadData1', 'PayloadData2']],
     caption="Process Termination Event - Timestamp in TimeCreated")

# Show full event for complete details
show(eventid_5, caption="Complete Process Termination Details")

if len(eventid_5) > 0:
    answer = eventid_5.iloc[0]['TimeCreated']
    executable = eventid_5.iloc[0]['ExecutableInfo']
    print(f"\n✓ Answer: {answer}")
    print(f"✓ Process terminated: {executable}")
    print("✓ Malicious process cleaned up after infection completed")

EventId 5 (Process Termination) events: 1
Only one event - answer immediately visible



Loading ITables v2.6.1 from the internet... (need help?)


Loading ITables v2.6.1 from the internet... (need help?)



✓ Answer: 2024-02-14 03:41:58.7996518
✓ Process terminated: C:\Users\CyberJunkie\Downloads\Preventivo24.02.14.exe.exe
✓ Malicious process cleaned up after infection completed
